# encoder-decoder-symmetric composite — cx6: Symmetric encoder (Conv/BN/LeakyReLU) and decoder (ConvT/BN/ReLU) stacks

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `encoder-decoder-symmetric`, `convtranspose-bn-activation-block`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "encoder-decoder-symmetric"
DD_ATOM_IDS = ["encoder-decoder-symmetric", "convtranspose-bn-activation-block"]
DD_SUBTOPICS = ["CNN: Encoder-decoder symmetric layout", "GAN: ConvT+BN+Activation block"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The DCGAN discriminator-as-encoder and generator-as-decoder share a structural contract: every downsampling step in the encoder is matched by an upsampling step in the decoder, and the two stacks are channel-mirrored.

1. **encoder-decoder-symmetric** — for every encoder layer `Conv2d(c, 2c, k=4, s=2, p=1)` (spatial /= 2, channels *= 2), the decoder has a mirrored `ConvTranspose2d(2c, c, k=4, s=2, p=1)` (spatial *= 2, channels /= 2). At the deepest point both stacks meet at the same shape.
2. **convtranspose-bn-activation-block** — the canonical *decoder*-side triple: `ConvT(k=4,s=2,p=1,bias=False) -> BN -> ReLU`. The encoder-side mirror uses `Conv2d(k=4,s=2,p=1,bias=False) -> BN -> LeakyReLU(0.2)` (DCGAN's published asymmetry).

**Anatomy (single encoder/decoder layer at one stage).**
```python
# Encoder block: spatial /= 2, channels *= 2.
enc_block = nn.Sequential(
    nn.Conv2d(c_in, c_out, 4, 2, 1, bias=False),
    nn.BatchNorm2d(c_out),
    nn.LeakyReLU(0.2, inplace=True),
)
# Decoder block (mirror): spatial *= 2, channels /= 2.
dec_block = nn.Sequential(
    nn.ConvTranspose2d(c_out, c_in, 4, 2, 1, bias=False),     # convtranspose-bn-activation-block.
    nn.BatchNorm2d(c_in),
    nn.ReLU(inplace=True),
)
```

**Why the activation asymmetry.** DCGAN found empirically that the discriminator trains more stably with LeakyReLU (no dead-neuron problem on negative inputs) while the generator wants crisp ReLU outputs. The *block structure* is symmetric — only the activation differs.

### Composite Exercise — Symmetric encoder (Conv/BN/LeakyReLU) and decoder (ConvT/BN/ReLU) stacks

**Atoms exercised together**: `encoder-decoder-symmetric`, `convtranspose-bn-activation-block`

Implement `cx6_build_symmetric_pair(channel_list)`. `channel_list` is a list like `[in_c, c1, c2, c3]` describing the encoder's channel progression: the encoder uses `Conv2d(channel_list[i], channel_list[i+1])` for each adjacent pair.

Return a tuple `(encoder, decoder)` of two `nn.Sequential`s:

- `encoder`: for each adjacent pair `(c_in, c_out)` in `channel_list`, append one block: `Conv2d(c_in, c_out, kernel_size=4, stride=2, padding=1, bias=False) -> BatchNorm2d(c_out) -> LeakyReLU(0.2, inplace=True)`.
- `decoder`: for each adjacent pair `(c_out, c_in)` in REVERSED `channel_list` (mirroring the encoder), append one block: `ConvTranspose2d(c_out, c_in, kernel_size=4, stride=2, padding=1, bias=False) -> BatchNorm2d(c_in) -> ReLU(inplace=True)` (atom: convtranspose-bn-activation-block, applied at each mirrored stage).

If `channel_list = [3, 16, 32]`:
- encoder has 2 blocks: `(3->16, BN16, LReLU)`, `(16->32, BN32, LReLU)`.
- decoder has 2 blocks: `(32->16, BN16, ReLU)`, `(16->3, BN3, ReLU)`.

Total Sequential children = `3 * (len(channel_list) - 1)` per stack (since each block contributes 3 layers).

The test checks:
- Both returned objects are `nn.Sequential`.
- Encoder Conv layers and Decoder ConvT layers are channel-mirrored (atom: encoder-decoder-symmetric).
- Encoder uses `LeakyReLU(negative_slope=0.2)`, decoder uses `ReLU`.
- All Conv/ConvT have `bias is None` (BN follows).
- Round-trip shape: `(B, in_c, H, W) -> encoder -> ... -> decoder -> (B, in_c, H, W)`. Use `H = W = 2**(len(channel_list) - 1)` so each `/2` stage stays integer.
- Round-trip works on a multiple-of-2 spatial size.

In [ ]:
def cx6_build_symmetric_pair(channel_list):
    # Atom A (encoder-decoder-symmetric): mirror channel pairs across the two stacks.
    enc_layers = []
    for c_in, c_out in zip(channel_list[:-1], channel_list[1:]):
        # Encoder-side block: Conv -> BN -> LeakyReLU (DCGAN discriminator side).
        enc_layers.append(nn.Conv2d(c_in, c_out, 4, 2, 1, bias=False))
        enc_layers.append(nn.BatchNorm2d(c_out))
        enc_layers.append(nn.LeakyReLU(0.2, inplace=True))
    encoder = nn.Sequential(*enc_layers)

    # Atom B (convtranspose-bn-activation-block) applied at each mirrored stage.
    dec_layers = []
    rev = list(reversed(channel_list))
    for c_out, c_in in zip(rev[:-1], rev[1:]):
        # Decoder-side block: ConvT -> BN -> ReLU (DCGAN generator side).
        dec_layers.append(nn.ConvTranspose2d(c_out, c_in, 4, 2, 1, bias=False))
        dec_layers.append(nn.BatchNorm2d(c_in))
        dec_layers.append(nn.ReLU(inplace=True))
    decoder = nn.Sequential(*dec_layers)

    return encoder, decoder


<details><summary>Show solution — cx6</summary>

```python
def cx6_build_symmetric_pair(channel_list):
    # Atom A (encoder-decoder-symmetric): mirror channel pairs across the two stacks.
    enc_layers = []
    for c_in, c_out in zip(channel_list[:-1], channel_list[1:]):
        # Encoder-side block: Conv -> BN -> LeakyReLU (DCGAN discriminator side).
        enc_layers.append(nn.Conv2d(c_in, c_out, 4, 2, 1, bias=False))
        enc_layers.append(nn.BatchNorm2d(c_out))
        enc_layers.append(nn.LeakyReLU(0.2, inplace=True))
    encoder = nn.Sequential(*enc_layers)

    # Atom B (convtranspose-bn-activation-block) applied at each mirrored stage.
    dec_layers = []
    rev = list(reversed(channel_list))
    for c_out, c_in in zip(rev[:-1], rev[1:]):
        # Decoder-side block: ConvT -> BN -> ReLU (DCGAN generator side).
        dec_layers.append(nn.ConvTranspose2d(c_out, c_in, 4, 2, 1, bias=False))
        dec_layers.append(nn.BatchNorm2d(c_in))
        dec_layers.append(nn.ReLU(inplace=True))
    decoder = nn.Sequential(*dec_layers)

    return encoder, decoder
```

The encoder/decoder asymmetry is JUST the activation (`LeakyReLU(0.2)` vs `ReLU`) and the conv direction (`Conv2d` vs `ConvTranspose2d`). Channel widths and kernel/stride/padding mirror exactly. `bias=False` is correct on both sides because BN immediately follows. With `k=4, s=2, p=1`, encoder Conv halves spatial size (`H_out = (H + 2*p - k) / s + 1 = (H - 2) / 2 + 1 = H/2`) and decoder ConvT doubles it (`H_out = (H - 1) * s - 2*p + k = 2*H`), so the round-trip is exact.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["CNN: Encoder-decoder symmetric layout", "GAN: ConvT+BN+Activation block"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()